# NuraVision: transfer learning de unas (NURA-105)

Notebook delgado: toda la logica vive en `services/vision/entrenamiento/`. Aqui solo se monta Drive, se instalan las versiones fijadas y se llama al modulo.

Requisitos: entorno con GPU (*Entorno de ejecucion -> Cambiar tipo -> GPU*). El dataset son fotos de manos de personas reales: vive solo en Drive, nunca se sube al repositorio ni se muestra aqui.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
TAREA = 'forma'  # 'forma', 'estado' o 'estado_clinico'
RAMA = 'feature/analisis-estado-clinico'  # la que tenga el codigo de la tarea elegida
DATASET = '/content/drive/MyDrive/nuravision/dataset'    # contiene <tarea>/<clase>/*.jpg
SALIDA = '/content/drive/MyDrive/nuravision/artifacts'   # aqui quedan modelos y reportes
# Obligatorio en estado_clinico: CSV archivo,serie con las fotos que son la misma una
# (README de services/vision, seccion 1). Vive en Drive, nunca en el repositorio.
SERIES = '/content/drive/MyDrive/nuravision/series_estado_clinico.csv'
ARG_SERIES = f'--series "{SERIES}"' if TAREA == 'estado_clinico' else ''

In [ ]:
!rm -rf /content/NuraVision-Page
!git clone -q --depth 1 -b {RAMA} https://github.com/Hyrum-Bustos-Duoc/NuraVision-Page.git /content/NuraVision-Page
%cd /content/NuraVision-Page/services/vision
!git rev-parse HEAD  # commit exacto de esta ejecucion: anotalo junto al reporte
!pip install -q -r requirements.txt

In [ ]:
# Los comandos siguientes corren en un proceso nuevo: usan las versiones recien instaladas.
!python -c "import sys, tensorflow as tf, keras; print(sys.version); print('tf', tf.__version__, 'keras', keras.__version__); print(tf.config.list_physical_devices('GPU'))"

Primero `dividir`: no entrena, solo valida fotos y muestra conteos por particion y rechazos. Revisalos antes de gastar GPU.

In [ ]:
!python -m entrenamiento dividir --tarea {TAREA} --dataset {DATASET} --salida {SALIDA} {ARG_SERIES}

In [ ]:
!python -m entrenamiento comparar --tarea {TAREA} --dataset {DATASET} --salida {SALIDA} {ARG_SERIES}

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{SALIDA}/{TAREA}/comparativa.md').read())